# Feature Engineering Factory
Run and cache various feature engineering methods for downstream models.


In [2]:
# Pre-load CUDA 13.x shared libs so bitsandbytes finds them regardless of LD_LIBRARY_PATH
import ctypes, os as _os
_cu13 = _os.path.join(_os.path.dirname(_os.__file__),
    'site-packages/nvidia/cu13/lib')
for _lib in ['libcudart.so.13', 'libcublas.so.13', 'libcublasLt.so.13', 'libnvJitLink.so.13']:
    try: ctypes.CDLL(_os.path.join(_cu13, _lib))
    except OSError: pass
del _cu13, _lib

# === ENVIRONMENT & FILEPATH SETUP ===
import os
import sys

codebase_path = "../"
if codebase_path not in sys.path:
    sys.path.insert(0, codebase_path)

# Clear src modules to allow reloading
for key in list(sys.modules.keys()):
    if key.startswith("src"):
        del sys.modules[key]

DATA_DIR = f"{codebase_path}data"
CACHE_DIR = f"{codebase_path}output/cache"
MODELS_DIR = f"{codebase_path}output/models"
ARTIFACTS_DIR = f"{codebase_path}artifacts"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [3]:
import os
import joblib
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

train = pd.read_csv(f"{DATA_DIR}/train.csv")
os.makedirs(CACHE_DIR, exist_ok=True)

### 1. TF-IDF Vectors
Sparse keyword vectors. Tunes max_features and n-grams.


In [ ]:
def extract_tfidf(train_df, max_features=5000, ngram_range=(1,1)):
    name = f"tfidf_max{max_features}_ng{ngram_range[0]}-{ngram_range[1]}"
    vec_path = f"{CACHE_DIR}/{name}_vectorizer.joblib"
    feat_path = f"{CACHE_DIR}/{name}_features.npz"
    
    if os.path.exists(feat_path) and os.path.exists(vec_path):
        print(f"Loading cached {name}...")
        return joblib.load(vec_path), sparse.load_npz(feat_path)
    
    print(f"Computing {name}...")
    vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=ngram_range, stop_words='english')
    X_sparse = vectorizer.fit_transform(train_df['query'])
    
    joblib.dump(vectorizer, vec_path)
    sparse.save_npz(feat_path, X_sparse)
    return vectorizer, X_sparse

# Run it
tfidf_vec, tfidf_X = extract_tfidf(train, max_features=30000, ngram_range=(1,3))
print("Shape:", tfidf_X.shape)

### 2. Truncated SVD (LSA)
Dimensionality reduction on TF-IDF to create dense topic vectors.


In [ ]:
def extract_svd(train_df, n_components=500, tfidf_max_features=5000, tfidf_ngram=(1,1)):
    name = f"svd_comp{n_components}_tfidf{tfidf_max_features}_ng{tfidf_ngram[0]}-{tfidf_ngram[1]}"
    svd_path = f"{CACHE_DIR}/{name}_model.joblib"
    feat_path = f"{CACHE_DIR}/{name}_features.npy"
    
    if os.path.exists(feat_path) and os.path.exists(svd_path):
        print(f"Loading cached {name}...")
        return joblib.load(svd_path), np.load(feat_path)
        
    print(f"Computing {name}...")
    _, tfidf_X = extract_tfidf(train_df, max_features=tfidf_max_features, ngram_range=tfidf_ngram)
    
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    X_dense = svd.fit_transform(tfidf_X)
    
    joblib.dump(svd, svd_path)
    np.save(feat_path, X_dense)
    return svd, X_dense

# Run it
svd_model, svd_X = extract_svd(train, n_components=1000, tfidf_max_features=30000, tfidf_ngram=(1,3))
print("Shape:", svd_X.shape)

### 3. Dense Semantic Embeddings (SentenceTransformers)
Captures semantic meaning using a pre-trained LLM.


In [ ]:
import os
import gc
import torch
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import BitsAndBytesConfig

def extract_dense_embeddings(train_df, model_name='all-MiniLM-L6-v2'):
    safe_name = model_name.replace("/", "-")
    name = f"dense_{safe_name}"
    feat_path = f"{CACHE_DIR}/{name}_features.npy"
    
    if os.path.exists(feat_path):
        print(f"Loading cached {name}...")
        return np.load(feat_path)
        
    print(f"Computing {name}... This may take a while.")
    
    # Force clear accumulated fragmented VRAM from previous crashes
    torch.cuda.empty_cache()
    gc.collect()
    
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True
    )
    
    # Use PyTorch's native SDPA instead of Flash Attention 2
    model = SentenceTransformer(
        model_name, 
        model_kwargs={
            "quantization_config": bnb_config,
            "attn_implementation": "sdpa"  # <--- FIXED: Uses native PyTorch optimization
        }
    )
    
    # Enforce sequence length limit to prevent OOM spikes on long texts
    model.max_seq_length = 2048  # <--- Set to 2048 or 4096 depending on text length
    
    X_dense = model.encode(
        train_df['query'].tolist(), 
        show_progress_bar=True,
        batch_size=4  
    )
    
    np.save(feat_path, X_dense)
    return X_dense

dense_X = extract_dense_embeddings(train, model_name='infgrad/Jasper-Token-Compression-600M') 
print("Shape:", dense_X.shape)


Computing dense_infgrad-Jasper-Token-Compression-600M... This may take a while.


modules.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.92k [00:00<?, ?B/s]

ModuleNotFoundError: No module named 'custom_st'

### 4. Handcrafted Meta-Features
Extracts numerical properties from raw text like length, code blocks, etc.


In [ ]:
def extract_meta_features(train_df, include_length=True, include_code_flags=True, include_math_flags=True):
    name = f"meta_len{int(include_length)}_code{int(include_code_flags)}_math{int(include_math_flags)}"
    feat_path = f"{CACHE_DIR}/{name}_features.npy"
    
    if os.path.exists(feat_path):
        print(f"Loading cached {name}...")
        return np.load(feat_path)
        
    print(f"Computing {name}...")
    features = []
    for q in train_df['query']:
        f = []
        if include_length:
            f.extend([len(q), len(str(q).split())])
        if include_code_flags:
            f.append(1 if '```' in str(q) or '`' in str(q) else 0)
        if include_math_flags:
            f.append(1 if '$' in str(q) or '\\' in str(q) else 0)
        features.append(f)
        
    X_meta = np.array(features)
    np.save(feat_path, X_meta)
    return X_meta

meta_X = extract_meta_features(train)
print("Shape:", meta_X.shape)



### 5. Custom Word2Vec (Averaged)
Trains domain-specific embeddings from scratch and averages them per query.


In [ ]:
def extract_word2vec(train_df, vector_size=500, window=10):
    name = f"w2v_size{vector_size}_win{window}"
    w2v_path = f"{CACHE_DIR}/{name}_model.joblib"
    feat_path = f"{CACHE_DIR}/{name}_features.npy"
    
    if os.path.exists(feat_path) and os.path.exists(w2v_path):
        print(f"Loading cached {name}...")
        return joblib.load(w2v_path), np.load(feat_path)
        
    print(f"Computing {name}...")
    from gensim.models import Word2Vec
    sentences = [str(q).lower().split() for q in train_df['query']]
    
    w2v = Word2Vec(sentences, vector_size=vector_size, window=window, min_count=1, workers=4)
    
    X_w2v = []
    for s in sentences:
        vecs = [w2v.wv[w] for w in s if w in w2v.wv]
        if len(vecs) > 0:
            X_w2v.append(np.mean(vecs, axis=0))
        else:
            X_w2v.append(np.zeros(vector_size))
            
    X_w2v = np.array(X_w2v)
    
    joblib.dump(w2v, w2v_path)
    np.save(feat_path, X_w2v)
    return w2v, X_w2v

# Note: Requires gensim to be installed
w2v_model, w2v_X = extract_word2vec(train)
print("Shape:", w2v_X.shape)